## Working with Databricks Model Serving in Pixeltable

Use `pxt.functions.databricks` to call **Foundation Models**, **Model Serving endpoints**, and **Mosaic AI agents** from computed columns.

### Prerequisites

- Databricks workspace with Model Serving or foundation model access
- Personal access token with serving permissions

### Important notes

- Databricks Model Serving usage may incur costs based on your workspace plan.
- Store PATs securely; prefer short-lived tokens or service principals in production.
- Be mindful of sensitive data sent to external model endpoints.

For SQL Warehouse import/export, Delta/Iceberg I/O, UC Volume storage, and hybrid deployment, see the full Databricks deployment guide (forthcoming).

First install required libraries and configure your Databricks credentials.

In [ ]:
%pip install -qU pixeltable openai

In [ ]:
import getpass
import os

if 'DATABRICKS_HOST' not in os.environ:
    os.environ['DATABRICKS_HOST'] = getpass.getpass(
        'Enter your Databricks workspace URL (e.g. https://<workspace>.cloud.databricks.com):'
    )

if 'DATABRICKS_TOKEN' not in os.environ:
    os.environ['DATABRICKS_TOKEN'] = getpass.getpass(
        'Enter your Databricks personal access token:'
    )

Now let's create a Pixeltable directory to hold the tables for our demo.

In [ ]:
import pixeltable as pxt
from pixeltable.functions import databricks as dbx_fn

pxt.drop_dir('dbx_demo', force=True)
pxt.create_dir('dbx_demo')

### Chat completions

Pass any foundation model name or deployed endpoint as `model` — including Mosaic AI agent endpoints.

In [ ]:
t = pxt.create_table('dbx_demo.docs', {'text': pxt.String})
msgs = [{'role': 'user', 'content': t.text}]

t.add_computed_column(
    output=dbx_fn.chat_completions(
        msgs,
        model='databricks-meta-llama-3-3-70b-instruct',
        model_kwargs={'max_tokens': 256, 'temperature': 0.2},
    )
)
t.add_computed_column(summary=t.output.choices[0].message.content)

# Deployed Mosaic AI agent:
# t.add_computed_column(
#     answer=dbx_fn.chat_completions(msgs, model='my-rag-agent-endpoint')
#         .choices[0].message.content
# )

In [ ]:
t.insert(
    [
        {
            'text': 'In one sentence, what is declarative data infrastructure?'
        }
    ]
)
t.collect()

### Embeddings and semantic search

Add embeddings and an embedding index, then run a similarity query.

In [ ]:
embed_fn = dbx_fn.embeddings.using(model='databricks-gte-large-en')
t.add_computed_column(
    embed=dbx_fn.embeddings(t.text, model='databricks-gte-large-en')
)
t.add_embedding_index('text', string_embed=embed_fn)

t.insert(
    [
        {
            'text': 'Machine learning is a subset of artificial intelligence.'
        },
        {'text': 'Deep learning uses neural networks with many layers.'},
        {'text': 'Python is a popular programming language.'},
    ]
)

sim = t.text.similarity(string='What is machine learning?')
t.order_by(sim, asc=False).limit(2).select(
    t.text, similarity=sim
).collect()